# Training CamemBERT NER Model for Travel Orders

Fine-tuning Jean-Baptiste/camembert-ner to distinguish DEPARTURE and DESTINATION entities

## 1. Setup and Configuration

In [ ]:
import torch
import json
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from sklearn.model_selection import train_test_split
from seqeval.metrics import f1_score, classification_report

# NER labels in BIO format
LABEL_LIST = [
    "O",              # Not an entity
    "B-DEPARTURE",    # Beginning of departure city
    "I-DEPARTURE",    # Inside departure city
    "B-DESTINATION",  # Beginning of destination city
    "I-DESTINATION",  # Inside destination city
    "B-TIME",         # Beginning of time expression
    "I-TIME",         # Inside time expression
]

label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for i, label in enumerate(LABEL_LIST)}

print("Label mappings:")
for label, idx in label2id.items():
    print(f"  {label}: {idx}")

## 2. Data Loading and Preparation

In [ ]:
# Load dataset from JSON file
with open('base/data/processed/travel-order-dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"Sample data format:")
print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

# Split into train/test (80/20)
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)

print(f"\nTraining samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")


# Define PyTorch Dataset Class
class NERDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):  # Reduced from 128 to 64
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]
        entities = item["entities"]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )
        
        labels = self._align_labels(text, entities, encoding)
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(labels)
        }
    
    def _align_labels(self, text, entities, encoding):
        """Align entity labels with tokens - IMPROVED VERSION"""
        offset_mapping = encoding["offset_mapping"].squeeze().tolist()
        labels = []
        
        # Track the first token index for each entity
        entity_first_token = {}  # Maps entity index to first token index
        
        for i, (start, end) in enumerate(offset_mapping):
            # Special tokens (CLS, SEP, PAD) have (0, 0) offsets
            if start == 0 and end == 0:
                labels.append(-100)
                continue
            
            # Find which entity this token belongs to
            token_label = "O"
            for ent_idx, entity in enumerate(entities):
                e_start, e_end = entity["start"], entity["end"]
                e_label = entity["label"]
                
                # IMPROVED: Check if token is FULLY within entity bounds
                # Use >= for start to include first character, < for end (exclusive)
                token_mid = (start + end) // 2  # Use midpoint for better overlap detection
                
                if e_start <= token_mid < e_end:
                    # Check if this is the first token for this entity
                    if ent_idx not in entity_first_token:
                        token_label = f"B-{e_label}"
                        entity_first_token[ent_idx] = i
                    else:
                        token_label = f"I-{e_label}"
                    break
            
            labels.append(label2id[token_label])
        
        return labels

print("\nDataset class defined successfully!")

In [59]:
# DEBUG: Check dataset quality
print("Checking first 5 training samples:")
for i, sample in enumerate(train_data[:5]):
    text = sample["text"]
    entities = sample["entities"]
    print(f"\n{i+1}. Text: {text}")
    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        entity_text = text[start:end]
        print(f"   - {label}: '{entity_text}' (chars {start}-{end})")
    
print(f"\n\nTotal samples in training: {len(train_data)}")
print(f"Total samples in test: {len(test_data)}")

# Check label distribution
from collections import Counter
label_counts = Counter()
for sample in train_data:
    for ent in sample["entities"]:
        label_counts[ent["label"]] += 1

print(f"\nLabel distribution in training data:")
for label, count in label_counts.items():
    print(f"  {label}: {count}")

Checking first 5 training samples:

1. Text: faut que jaille a aubergenville élisabethville depuis drancy
   - DESTINATION: 'aubergenville élisabethville' (chars 18-46)
   - DEPARTURE: 'drancy' (chars 54-60)

2. Text: Cernay (Val-d'Oise) en provenance de Antibes
   - DESTINATION: 'Cernay (Val-d'Oise)' (chars 0-19)
   - DEPARTURE: 'Antibes' (chars 37-44)

3. Text: je voudrais rejoindre Nemours - Saint-Pierre depuis Dax
   - DESTINATION: 'Nemours - Saint-Pierre' (chars 22-44)
   - DEPARTURE: 'Dax' (chars 52-55)

4. Text: on part de plaisir - grignon pour aller a mantes-la-jolie
   - DEPARTURE: 'plaisir - grignon' (chars 11-28)
   - DESTINATION: 'mantes-la-jolie' (chars 42-57)

5. Text: comment je me déplace à les aubrais de la verrière
   - DESTINATION: 'les aubrais' (chars 24-35)
   - DEPARTURE: 'la verrière' (chars 39-50)


Total samples in training: 400
Total samples in test: 100

Label distribution in training data:
  DESTINATION: 400
  DEPARTURE: 400


In [60]:
# DEBUG: Check how labels are aligned during training
print("Testing label alignment on a sample:")
sample = train_data[0]
text = sample["text"]
entities = sample["entities"]

print(f"Text: {text}")
print(f"Entities: {entities}")

# Manually test the alignment
test_dataset = NERDataset([sample], tokenizer)
item = test_dataset[0]

# Get the tokens
tokens = tokenizer.convert_ids_to_tokens(item["input_ids"].tolist())
labels = item["labels"].tolist()

print(f"\nTokenization and labels:")
for i, (token, label_id) in enumerate(zip(tokens[:30], labels[:30])):
    if label_id != -100:
        label_name = id2label[label_id]
        print(f"  {i}: '{token}' -> {label_name} (label_id={label_id})")
    else:
        print(f"  {i}: '{token}' -> [IGNORED] (label_id=-100)")

# Check if any entity labels are present
entity_label_count = sum(1 for l in labels if l > 0)
print(f"\n✅ Entity labels found: {entity_label_count}")
print(f"✅ Total non-padding labels: {sum(1 for l in labels if l != -100)}")

if entity_label_count == 0:
    print("\n❌ WARNING: No entity labels found! This means the label alignment is broken.")
    print("   The model will not learn to detect entities.")

Testing label alignment on a sample:
Text: faut que jaille a aubergenville élisabethville depuis drancy
Entities: [{'start': 18, 'end': 46, 'label': 'DESTINATION'}, {'start': 54, 'end': 60, 'label': 'DEPARTURE'}]

Tokenization and labels:
  0: '<s>' -> [IGNORED] (label_id=-100)
  1: '▁faut' -> O (label_id=0)
  2: '▁que' -> O (label_id=0)
  3: '▁j' -> O (label_id=0)
  4: 'aille' -> O (label_id=0)
  5: '▁a' -> O (label_id=0)
  6: '▁auberge' -> B-DESTINATION (label_id=3)
  7: 'n' -> I-DESTINATION (label_id=4)
  8: 'ville' -> I-DESTINATION (label_id=4)
  9: '▁' -> I-DESTINATION (label_id=4)
  10: 'éli' -> I-DESTINATION (label_id=4)
  11: 's' -> I-DESTINATION (label_id=4)
  12: 'a' -> I-DESTINATION (label_id=4)
  13: 'beth' -> I-DESTINATION (label_id=4)
  14: 'ville' -> I-DESTINATION (label_id=4)
  15: '▁depuis' -> O (label_id=0)
  16: '▁d' -> B-DEPARTURE (label_id=1)
  17: 'r' -> I-DEPARTURE (label_id=2)
  18: 'ancy' -> I-DEPARTURE (label_id=2)
  19: '</s>' -> [IGNORED] (label_id=-100)
  2

## 3. Model and Training Setup

In [61]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = []
    pred_labels = []
    
    for pred_seq, label_seq in zip(predictions, labels):
        true_seq = []
        pred_seq_labels = []
        for pred, label in zip(pred_seq, label_seq):
            if label != -100:
                true_seq.append(id2label[label])
                pred_seq_labels.append(id2label[pred])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_labels)
    
    return {"f1": f1_score(true_labels, pred_labels)}


# Load pre-trained French NER model
model_name = "Jean-Baptiste/camembert-ner"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)



# Prepare datasets
train_dataset = NERDataset(train_data, tokenizer)
test_dataset = NERDataset(test_data, tokenizer)
data_collator = DataCollatorForTokenClassification(tokenizer)


# Configure training with optimized settings for faster training
training_args = TrainingArguments(
    num_train_epochs=10,              # Reduced from 30 to 10 (saves ~67% time)
    per_device_train_batch_size=16,   # Increased from 8 to 16 (saves ~30% time)
    per_device_eval_batch_size=16,
    learning_rate=3e-5,                # Slightly higher LR for faster convergence
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=10,
    warmup_ratio=0.1,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),    # Use mixed precision if GPU available
    # Early stopping with EarlyStoppingCallback (optional, add if needed)
)

print(f"Training with {model_name} on TravelOrder dataset with optimized settings:")
print(f"  Epochs: {training_args.num_train_epochs} (reduced from 30)")
print(f"  Batch size: {training_args.per_device_train_batch_size} (increased from 8)")
print(f"  Learning rate: {training_args.learning_rate}")


# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Training with Jean-Baptiste/camembert-ner on TravelOrder dataset with optimized settings:
  Epochs: 10 (reduced from 30)
  Batch size: 16 (increased from 8)
  Learning rate: 3e-05


C:\Users\Yanis\AppData\Local\Temp\ipykernel_2072\1224969225.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 4. Training and Saving

In [ ]:
# Train the model
print("Starting training...\n")
trainer.train()

# Save the model
model_save_path = "base/models/BERT/camembert-ner-travel"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"\n✅ Model saved to: {model_save_path}")

## 5. Evaluation and Testing

In [ ]:
# Define prediction function
def predict(text, model, tokenizer):
    """Extract DEPARTURE, DESTINATION and TIME entities from text"""
    model.eval()
    device = next(model.parameters()).device
    
    # Use the model's id2label mapping
    model_id2label = model.config.id2label
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        padding=True,
        truncation=True,
        max_length=128
    )
    
    offset_mapping = inputs.pop("offset_mapping").squeeze().tolist()
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()
    
    # Ensure predictions is a list (in case of single token)
    if not isinstance(predictions, list):
        predictions = [predictions]
    
    # Extract entities
    entities = {"DEPARTURE": [], "DESTINATION": [], "TIME": []}
    current_entity = None
    current_tokens = []
    
    for i, (pred, (start, end)) in enumerate(zip(predictions, offset_mapping)):
        if start == 0 and end == 0:
            continue
        
        # Convert prediction index to label string (use integer keys)
        label = model_id2label.get(pred, "O")
        
        if label.startswith("B-"):
            if current_entity and current_tokens:
                entity_text = text[current_tokens[0][0]:current_tokens[-1][1]].strip()
                entities[current_entity].append(entity_text)
            
            current_entity = label[2:]
            current_tokens = [(start, end)]
            
        elif label.startswith("I-") and current_entity == label[2:]:
            current_tokens.append((start, end))
            
        else:
            if current_entity and current_tokens:
                entity_text = text[current_tokens[0][0]:current_tokens[-1][1]].strip()
                entities[current_entity].append(entity_text)
            current_entity = None
            current_tokens = []
    
    if current_entity and current_tokens:
        entity_text = text[current_tokens[0][0]:current_tokens[-1][1]].strip()
        entities[current_entity].append(entity_text)
    
    return entities

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_dataset)

print("Test Results:")
for key, value in test_results.items():
    print(f"  {key}: {value}")


z:\Code - Programmation\Epitech\IA\AIA-911\TravelOrder\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Results:
  eval_loss: 0.020492129027843475
  eval_f1: 0.9702970297029702
  eval_runtime: 5.3832
  eval_samples_per_second: 18.576
  eval_steps_per_second: 1.3
  epoch: 10.0


In [ ]:
model_path = "base/models/BERT/camembert-ner-travel"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# IMPORTANT: Move model to device and set to eval mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Verify the label mappings are correct
print("Loaded model config:")
print(f"  id2label from model: {model.config.id2label}")
print(f"  Using device: {device}")
print()

# Test with real examples
test_sentences = [
    "Je veux aller de Paris à Marseille",
    "Un billet de Lyon pour Bordeaux s'il vous plaît",
    "Je dois partir de Toulouse vers Nice demain",
    "Trajet entre Nantes et Strasbourg lundi prochain",
    "Je cherche un train de Montpellier à Lille à 14h",
    "Horaires des TGV de Rennes à La Rochelle ce vendredi",
    "Je pars de Clermont-Ferrand direction Aix-en-Provence dans l'après-midi",
    "Y a-t-il un train de Grenoble à Genève ce soir ?",
]

print("\n" + "=" * 70)
print("Testing model on real examples:")
print("=" * 70)

for sentence in test_sentences:
    entities = predict(sentence, model, tokenizer)
    departure = entities['DEPARTURE'][0] if entities['DEPARTURE'] else "Not found"
    destination = entities['DESTINATION'][0] if entities['DESTINATION'] else "Not found"
    time = entities['TIME'][0] if entities['TIME'] else "Not found"
    
    print(f"\n{sentence}")
    print(f"  Departure: {departure}")
    print(f"  Destination: {destination}")
    print(f"  Time: {time}")

print("\n" + "=" * 70)